In [1]:
import torch 
import numpy as np 
import h5py
import os
import IPython.display as ipd
from corpus.speaker_room_dataset import SpeakerRoomDataset
import src.audio_transforms as at

import src.spatial_attn_lightning as binaural_lightning 
import yaml
from tqdm.auto import tqdm

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

torch.set_float32_matmul_precision('medium')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

%matplotlib inline
import matplotlib.pyplot as plt

from IPython.display import Audio, display

/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!hostname

node100


In [3]:
### Get most recent config
# config_path = "config/binaural_attn/word_task_25p_loc_v07_LN_last_valid_time_no_affine.yaml"
# ckpt_path = "attn_cue_models/word_task_25p_loc_v07_LN_last_valid_time_no_affine/checkpoints/epoch=3-step=49432.ckpt"
# old_style = False
### Get most recent config
config_path = "config/binaural_attn/word_task_v10_main_feature_gain_config.yaml"
ckpt_path = "attn_cue_models/word_task_v10_main_feature_gain_config/checkpoints/epoch=1-step=24679-v1.ckpt"
# old_style = True 

config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['hparas']['batch_size'] = 1 # config['data']['loader']['batch_size'] // args.gpus
config['num_workers'] = 2
config['noise_kwargs']['low_snr'] = 0
config['noise_kwargs']['high_snr'] = 0
# get model input sr for brir resampling
signal_sr = config['audio']['rep_kwargs']['sr']
coch_sr = config['audio']['rep_kwargs']['env_sr']
# cue_duration = 0.5
# n_cue_frames = int(cue_duration * signal_sr)
# config['model']['n_cue_frames'] = n_cue_frames

# config['cue_duration_test'] = True


In [4]:
model = binaural_lightning.BinauralAttentionModule.load_from_checkpoint(checkpoint_path=ckpt_path, config=config).cuda().eval()

Using explicit dim specification for demeaning in audio transforms
Using BinauralAuditoryAttentionCNN
v08 True
num_classes={'num_words': 800}
Model performing word task
Using singe gain function per layer
Conv block order: LN -> Conv -> ReLU
fc_attn: True
coch_affine: True
Compiling model...


/om2/user/imgriff/conda_envs/pytorch_2/lib/python3.11/site-packages/torchaudio/functional/functional.py:1371: UserWarning: "kaiser_window" resampling method name is being deprecated and replaced by "sinc_interp_kaiser" in the next release. The default behavior remains unchanged.
  warnings.warn(


center_crop=True
binaural=True
Binaural cochleagram
using FIR cochleagram


## Define alternate audio transforms to see effect on generated distractors

In [11]:

def nonzero_demean(x):
    return x - x[x.nonzero(as_tuple=True)].ravel().mean()
    
def nonzero_rms(x):
    return torch.sqrt(torch.mean(torch.pow(x[x.nonzero(as_tuple=True)].ravel().mean(), 2)))

def rms(x):
    return torch.sqrt(torch.mean(torch.pow(x.ravel(), 2)))


def rms_to_db(rms):
    return 20 * np.log10(rms / 20e-6)




In [12]:
standard_audio_transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.BinauralCombineWithRandomDBSNR(low_snr=-6,
                                                  high_snr=-6,
                                                  v2_demean=True),
                at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02, v2_demean=True), # 20 * np.log10(0.02/20e-6) = 60 dB SPL 
            ])

rms_norm =  at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02, v2_demean=True)
# nonzero_audio_transforms = at.AudioCompose([
#                 at.AudioToTensor(),
#                 BinauralCombineWithRandomDBSNRNonZeros(snr=0),
#                 at.BinauralRMSNormalizeForegroundAndBackground(rms_level=0.02, v2_demean=True), # 20 * np.log10(0.02/20e-6) = 60 dB SPL 
#             ])




In [13]:
import pickle
class_map = pickle.load( open("/om2/user/imgriff/datasets/commonvoice_9/en/cv_800_word_label_to_int_dict.pkl", "rb" )) 
ix_to_word = {v: k for k, v in class_map.items()}

### Get spatialization fns

In [14]:
from pathlib import Path 
import pandas as pd 
import soxr 
room_ix = 0 
test_IR_manifest_dir = Path("/om2/user/imgriff/spatial_audio_pipeline/assets/brir/mit_bldg46room1004_min_reverb")
test_IR_manifest_path = test_IR_manifest_dir / "manifest_brir.pdpkl"
h5_fn = test_IR_manifest_dir / f"room{room_ix:04}.hdf5"
new_room_manifest = pd.read_pickle(test_IR_manifest_path)
only14_manifest = new_room_manifest[(new_room_manifest['index_room'] == room_ix)  & (new_room_manifest['src_dist'] == 1.4)]

def get_brir(azim=None, elev=None, coords=None, h5_fn=None, IR_df=None, out_sr=44_100):
    if coords is not None:
        azim, elev = coords
    df_row = IR_df[(IR_df['src_azim'] == azim) & (IR_df['src_elev'] == elev)]
    brir_ix = df_row['index_brir'].values[0]
    sr_src = df_row['sr'].values[0]
    with h5py.File(h5_fn, 'r') as f:
        brir = f['brir'][brir_ix]
    if out_sr != sr_src:
        brir = soxr.resample(brir.astype(np.float32), sr_src, out_sr)
    return brir

## Distractor optimization 

In [9]:

import torch.fft as fft


def lowpass_torch(input, limit, sr=44_100):
    device = input.device
    kernel = torch.abs(fft.rfftfreq(input.shape[-1], d=1/sr, dtype=input.dtype)) < limit
    fft_input = fft.rfft(input)
    return fft.irfft(fft_input * kernel.to(device), n=input.shape[-1])


class WeightedFFT(torch.nn.Module):
    def __init__(self, n_freqs, sr=44_100, cutoff_freq=12_000):
        super().__init__()
        self.n_freqs = n_freqs
        freq_weights = torch.ones(n_freqs)
        self.freqs = torch.abs(fft.rfftfreq(2*n_freqs-1, d=1/sr, requires_grad=False))
        freq_weights[self.freqs > cutoff_freq] = 0
        self.weights = torch.nn.Parameter(freq_weights)
        # self.cutoff_freq = cutoff_freq
    
    def forward(self, x):
        fft_input = fft.rfft(x)
        return fft.irfft(fft_input * self.weights, n=x.shape[-1])

class FFTFreqConstraint(object):
    def __init__(self, cutoff_freq=12_000):
        self.cutoff_freq = cutoff_freq

    def __call__(self, module):
        if hasattr(module,'weight'):
            w = module.weight.data
            # set weights above cutoff_freq to 0
            w[module.freqs > self.cutoff_freq] = 0
            module.weight.data = w.clamp(0,1)


def tanhclip(x, a, b):
    """Canonical soft-clipping with tanh(x).  Must specify both endpoints"""
    scale = (b - a) / 2
    return (torch.tanh((x - a) / scale - 1) + 1) * scale + a


def set_dbspl(x, dbspl, mean_subtract=True, axis=-1, strict=True):
    """
    Set sound pressure level in dB re 20e-6 Pa (dB SPL) of a tensor.
    
    Args
    ----
    x (tensor): tensor for which sound pressure level is set
    dbspl (tensor): desired sound pressure level in dB re 20e-6 Pa
    mean_subtract (bool): if True, x is first de-meaned
    axis (int or list): axis along which to measure RMS amplitudes
    strict (bool): if True, an error will be raised if x is silent;
        if False, silent signals will be returned as-is
    
    Returns
    -------
    out (tensor): sound pressure level of x in dB re 20e-6 Pa
    """
    if mean_subtract:
        x = x - x.ravel().mean() 
    rms_new = 20e-6 * np.power(10.0, dbspl / 20.0)
    rms_old = at.ch_rms(x.ravel()) 

    out = torch.mul(rms_new / rms_old, x)
    return out


class WindowRMSEquate(torch.nn.Module):
    def __init__(self, win_size_s, rms_op, sr=44100):
        super().__init__()
        self.win_size = int(win_size_s * sr )
        self.rms_op = rms_op 
        self.sr=sr

    def forward(self, x):
        in_shape = x.shape
        x = x - x.ravel().mean()
        x = x.view(-1,self.win_size)
        x, _ = self.rms_op(x, None)
        return x.view(in_shape)

    

## New proceedure: Optimize background sound that gives masking release

In [15]:
dataset = SpeakerRoomDataset(manifest_path='/om2/user/rphess/Auditory-Attention/final_binaural_manifest.pkl',
                            excerpt_path='/om2/user/msaddler/spatial_audio_pipeline/assets/swc/manifest_all_words.pdpkl',
                            cue_type='voice_and_location',
                            sr=signal_sr) 


In [18]:
## dev dummy signal optimization proceedure - will most likely yeild adversarial example 
# from torch.optim.lr_scheduler import OneCycleLR
from torchaudio.functional import highpass_biquad, lowpass_biquad, bandpass_biquad

RANDOMSEED=12
NOISESCALE = 1e-6
EARLYSTOP = 200
SAMPLERATE= 44_100
cutoff_freq = 8_000

win_size_s = .1

torch.manual_seed(RANDOMSEED)
np.random.seed(RANDOMSEED)


audio_transforms = standard_audio_transforms.cuda()
rms_equate = WindowRMSEquate(win_size_s, rms_norm, sr=SAMPLERATE)

target_brir = get_brir(azim=0, elev=0, h5_fn=h5_fn, IR_df=only14_manifest, out_sr=SAMPLERATE)
sp_to_target_loc = at.Spatialize(target_brir, model_sr=signal_sr).cuda()

bg_brir = get_brir(azim=0, elev=0, h5_fn=h5_fn, IR_df=only14_manifest, out_sr=SAMPLERATE)
sp_to_bg_loc = at.Spatialize(bg_brir, model_sr=signal_sr).cuda()


coch_transform = model.coch_gram.cuda()
rms_norm = rms_norm.cuda()


ix = 30

cue, fg, bg, label, confusion = dataset[ix]
distractor = (torch.rand_like(bg) * NOISESCALE).cuda()

cue = sp_to_target_loc(cue.unsqueeze(0).cuda())
fg = sp_to_target_loc(fg.unsqueeze(0).cuda())
bg = sp_to_target_loc(bg.unsqueeze(0).cuda())
distractor = sp_to_bg_loc(distractor.unsqueeze(0).cuda())

cue, _ = audio_transforms(cue, None)
mixture, _ = audio_transforms(fg, bg)


distractor, _ = audio_transforms(distractor, None)

label = torch.tensor(label).unsqueeze(0).cuda() 
# init as noise 

distractor.requires_grad_(True)
distractor.retain_grad()


optimizer = torch.optim.AdamW(
            [distractor], #weighted_fft.weights],
            lr=0.0001,
            # momentum=0.9,
            # weight_decay=5e-4,
        )

loss_fn =  torch.nn.CrossEntropyLoss()

n_steps = 1000


best_loss = 100
best_distractor = None
best_mixture = None 
best_step = 0 
early_stop = EARLYSTOP

# cue_cg, _ = coch_transform(cue, None)
mixture_cg, _ = coch_transform(mixture, None)

for step in (pbar := tqdm(range(n_steps))):
    optimizer.zero_grad()

    # make rms equal in windows for distractor
    # distractor_lp = lowpass_torch(distractor, cutoff_freq, sr=SAMPLERATE)
    # distractor_eq = rms_equate(distractor)
    dist_wav, _ = audio_transforms(distractor, None)
    # mixture_cg, dist_cg = coch_transform(mixture_wav, distractor_eq)
    dist_cg, _ = coch_transform(dist_wav, None)

    logits = model(dist_cg, mixture_cg, None)


    loss = loss_fn(logits, label)  # -  (10 * corr) 

 
    # enforce difference between distractor and fg in cochleagram 
    # loss -= (fg_cg - dist_cg).flatten().norm(p=2)

    if loss < best_loss:
        best_loss = loss.detach().item()
        # print(f"best loss {best_loss.detach().item():.3f} on step {step}")
        best_distractor = distractor.detach()
        # best_mixture = mixture_wav.detach()
        best_step = step 
        early_stop = EARLYSTOP
        pbar.set_postfix_str(f"Best loss: {best_loss:.4f} Current loss: {loss.detach().item():.4f} Early stop: {early_stop}")#HP energy: {hp_energy.detach().item():.3f}")

    else:
        early_stop -= 1 

    # if step % 100 == 0: 
    pbar.set_postfix_str(f"Best loss: {best_loss:.4f} Current loss: {loss.detach().item():.4f} Early stop: {early_stop}")#HP energy: {hp_energy.detach().item():.3f}")

    if early_stop == 0:
        break 
    
    # apply constraints
    # weighted_fft.apply(fft_constraint)
    # clip distractor 
    # distractor.data = torch.clamp(distractor, -1e-3, 1e-3)
    
    loss.backward()
    optimizer.step()

100%|██████████| 1000/1000 [02:26<00:00,  6.82it/s, Best loss: 0.0019 Current loss: 0.0019 Early stop: 200]


In [19]:
mixture.cpu().shape

torch.Size([1, 2, 110250])

In [20]:
print(ix_to_word[label.item()])

industry


In [22]:
print("Cue")
display(Audio(cue.cpu().squeeze(), rate=SAMPLERATE))
print("Mixture")
display(Audio(mixture.cpu().squeeze(), rate=SAMPLERATE))
print("Gen. Cue")
display(Audio(best_distractor.detach().cpu().squeeze(), rate=SAMPLERATE))

Cue


Mixture


Gen. Cue


In [21]:
print("Cue")
display(Audio(cue.cpu().squeeze(), rate=SAMPLERATE))
print("Mixture")
display(Audio(mixture.cpu().squeeze(), rate=SAMPLERATE))
print("Denoised Mixture")
display(Audio(mixture_wav.detach().cpu().squeeze(), rate=SAMPLERATE))

Cue


Mixture


Denoised Mixture


NameError: name 'mixture_wav' is not defined